# Desafio: Agente de IA para Fintech / Banco Digital (OCI + LangChain)

Este notebook implementa um Agente de Atendimento e Consulta Baseado em Documentos para um **Banco Digital / Fintech**, alimentado com:
- Política de privacidade e proteção de dados (LGPD)
- Termos e condições de uso
- Perguntas frequentes sobre transações e limites (Pix, TED, boletos)
- Política de segurança e prevenção a fraudes
- Tarifas e comissões do serviço

---

## 1. Setup e Instalação de Dependências

In [ ]:
%pip install -q -U langchain langchain-community oci pypdf pandas faiss-cpu "numpy<2" "scipy<1.13"

Note: you may need to restart the kernel to use updated packages.


## 2. Configuração de Autenticação Oracle Cloud Infrastructure (OCI)

Defina as variáveis de ambiente necessárias para comunicação com o serviço **OCI Generative AI**.

In [17]:
import os

# Configuração dos parâmetros de acesso na OCI
# Caso rodando na VM Compute da OCI com permissão atribuída, altere auth_type para INSTANCE_PRINCIPAL
OS_COMPARTMENT_ID = os.getenv('OCI_COMPARTMENT_ID', 'ocid1.compartment.oc1..seu_compartment_ocid')
OCI_ENDPOINT = os.getenv('OCI_GENAI_ENDPOINT', 'https://inference.generativeai.us-chicago-1.oci.oraclecloud.com')

print('Parâmetros OCI configurados com sucesso.')

Parâmetros OCI configurados com sucesso.


## 3. Processamento dos Documentos Institucionais da Fintech

In [18]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# Carregamento de todos os PDFs regulatórios da Fintech
loader = DirectoryLoader('documentos_fintech/', glob='*.pdf', loader_cls=PyPDFLoader)
documents = loader.load()

# Fragmentação dos documentos preservando contexto normativo
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", " ", ""]
)
docs_processed = text_splitter.split_documents(documents)

print(f'{len(docs_processed)} blocos normativos processados para a base do Banco Digital.')

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 4. Construção do Agente com OCI Generative AI

In [ ]:
from langchain_community.llms import OCIGenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Inicialização do serviço Generative AI da OCI
llm = OCIGenAI(
    model_id="cohere.command-r-plus",
    service_endpoint=OCI_ENDPOINT,
    compartment_id=OS_COMPARTMENT_ID,
    model_kwargs={"temperature": 0.1, "max_tokens": 600},
    auth_type="API_KEY"  # Altere para INSTANCE_PRINCIPAL na VM OCI Compute
)

# System Prompt estritamente regulado para Fintech
system_prompt = """
Você é o assistente virtual oficial de atendimento da Fintech / Banco Digital.
Sua função é responder dúvidas dos clientes com precisão, cordialidade e tom profissional.

REGRAS DE CONDUTA E SEGURANÇA:
1. Responda APENAS com base nos documentos regulatórios fornecidos no contexto abaixo.
2. Se a informação sobre limites, tarifas, fraudes ou privacidade não estiver explicitada no contexto, informe categoricamente que não possui essa informação e oriente o cliente a entrar em contato com o suporte humano.
3. NUNCA solicite ou confirme dados sensíveis do cliente (como senhas, CVV do cartão ou tokens Pix) no chat.

Contexto Regulatório:
{contexto}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{pergunta}")
])

chain = prompt | llm | StrOutputParser()

def consultar_banco_digital(pergunta, chunks):
    contexto = "\n\n---\n\n".join([doc.page_content for doc in chunks[:4]])
    return chain.invoke({"contexto": contexto, "pergunta": pergunta})

# Exemplo de consulta ao agente
pergunta = "Qual é a tarifa para saque no Banco24Horas e qual o limite do Pix noturno?"
print("Pergunta:", pergunta)
print("Pronto para execução do agente.")

## 5. Guia de Implantação no OCI Compute

### Checklist para o Ambiente de Produção:
1. **Segurança (VCN & NSG):** Configurar Security List da VCN na OCI liberando tráfego na porta `8000` (FastAPI) ou `8501` (Streamlit).
2. **Identidade (IAM):** Criar *Dynamic Group* para a VM Compute e conceder permissão na política de IAM:
   `ALLOW DYNAMIC-GROUP Grupo-VM-Fintech TO USE genai-agent-family IN COMPARTMENT id_compartment`
3. **Execução:** Subir a API utilizando `auth_type="INSTANCE_PRINCIPAL"` sem necessidade de expor chaves no código.